In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 10


In [ ]:
DATASET_DIR = "/kaggle/input/macular-degeneration-disease-dataset/AMDNet23 Fundus Image Dataset for Age-Related Macular Degeneration Disease Detection/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset"
print(os.listdir(DATASET_DIR))

TRAIN_DIR = f"{DATASET_DIR}/train"
VAL_DIR   = f"{DATASET_DIR}/valid"




In [ ]:
from pathlib import Path
import os
from PIL import Image

# ⚠️ 重點：這裡「不再定義 DATASET_DIR」
# 直接使用 Step 2 已經驗證過的 DATASET_DIR

SRC_TRAIN = Path(DATASET_DIR) / "train"
SRC_VALID = Path(DATASET_DIR) / "valid"

CLEAN_ROOT = Path("/kaggle/working/clean_dataset")
CLEAN_TRAIN = CLEAN_ROOT / "train"
CLEAN_VALID = CLEAN_ROOT / "valid"

ext_ok = {".jpg", ".jpeg", ".png"}

def make_clean_split(src_split: Path, dst_split: Path):
    dst_split.mkdir(parents=True, exist_ok=True)
    linked = 0
    skipped = 0

    for cls_dir in src_split.iterdir():
        if not cls_dir.is_dir():
            continue

        (dst_split / cls_dir.name).mkdir(parents=True, exist_ok=True)

        for fp in cls_dir.iterdir():
            if not fp.is_file():
                continue

            if fp.suffix.lower() not in ext_ok:
                skipped += 1
                continue

            # 驗證是否為可讀圖片（避免壞檔）
            try:
                with Image.open(fp) as im:
                    im.verify()
            except Exception:
                skipped += 1
                continue

            dst = dst_split / cls_dir.name / fp.name
            if not dst.exists():
                os.symlink(fp, dst)

            linked += 1

    return linked, skipped


train_linked, train_skipped = make_clean_split(SRC_TRAIN, CLEAN_TRAIN)
valid_linked, valid_skipped = make_clean_split(SRC_VALID, CLEAN_VALID)

print("Train linked:", train_linked, "skipped:", train_skipped)
print("Valid linked:", valid_linked, "skipped:", valid_skipped)


In [ ]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLEAN_TRAIN),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLEAN_VALID),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# ⭐ 先存 class_names
class_names = train_ds.class_names

# ⭐ 再忽略壞檔
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds   = val_ds.apply(tf.data.experimental.ignore_errors())


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 0：Baseline
# =====================
model_0 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dense(4, activation="softmax")
])

model_0.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_0 = model_0.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 正確評估流程（關鍵修正）
# =====================

# 重新建立「不 shuffle」的 validation dataset（只用來評估）
val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

# 預測
y_pred = np.argmax(model_0.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

# 輸出結果
print("\n📌 Model 0 (Baseline)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 1：Larger CNN
# =====================
model_1 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dense(4, activation="softmax")
])

model_1.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_1 = model_1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 正確評估流程（與 Model 0 完全一致）
# =====================

# 確保 VAL_DIR 已定義
# VAL_DIR = ".../AMDNet23 Dataset/valid"

val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

# 預測與真實標籤
y_pred = np.argmax(model_1.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

# 輸出結果
print("\n📌 Model 1 (Larger CNN)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 2：Smaller CNN
# =====================
model_2 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(8, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dense(4, activation="softmax")
])

model_2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_2 = model_2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 正確評估流程（與 Model 0 / Model 1 完全一致）
# =====================

# 確保 VAL_DIR 已定義
# VAL_DIR = ".../AMDNet23 Dataset/valid"

val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

# 預測與真實標籤
y_pred = np.argmax(model_2.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

# 輸出結果
print("\n📌 Model 2 (Smaller CNN)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 3：With Dropout
# =====================
model_3 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),          # ⭐ Dropout 實驗重點
    layers.Dense(4, activation="softmax")
])

model_3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_3 = model_3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 評估（與其他模型一致）
# =====================
val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

y_pred = np.argmax(model_3.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

print("\n📌 Model 3 (With Dropout)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

# =====================
# ⭐ Loss & Accuracy 曲線（報告用）
# =====================
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history_3.history["accuracy"], label="Train Accuracy")
plt.plot(history_3.history["val_accuracy"], label="Val Accuracy")
plt.title("Model 3 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history_3.history["loss"], label="Train Loss")
plt.plot(history_3.history["val_loss"], label="Val Loss")
plt.title("Model 3 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 4：Larger Dense Layer
# =====================
model_4 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),   # ⭐ Dense 層變大（實驗重點）
    layers.Dense(4, activation="softmax")
])

model_4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_4 = model_4.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 正確評估（與 Model 0–3 完全一致）
# =====================

val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

y_pred = np.argmax(model_4.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

print("\n📌 Model 4 (Larger Dense Layer)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score

# =====================
# Model 5：With Data Augmentation
# =====================
model_5 = models.Sequential([
    layers.Rescaling(1./255, input_shape=(128,128,3)),

    # ⭐ 資料增強（只在訓練階段作用）
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dense(4, activation="softmax")
])

model_5.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------- 訓練 ----------
history_5 = model_5.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# =====================
# ⭐ 評估（與其他模型一致）
# =====================
val_ds_eval = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(128,128),
    batch_size=16,
    shuffle=False
)

y_pred = np.argmax(model_5.predict(val_ds_eval), axis=1)
y_true = np.concatenate([y for x, y in val_ds_eval], axis=0)

print("\n📌 Model 5 (With Data Augmentation)")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))

# =====================
# ⭐ Loss & Accuracy 曲線（報告用）
# =====================
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history_5.history["accuracy"], label="Train Accuracy")
plt.plot(history_5.history["val_accuracy"], label="Val Accuracy")
plt.title("Model 5 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history_5.history["loss"], label="Train Loss")
plt.plot(history_5.history["val_loss"], label="Val Loss")
plt.title("Model 5 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()
